<a href="https://colab.research.google.com/github/ChrisHuang7260/TrialWatchAIHackathon_Track2/blob/main/demo_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U bitsandbytes accelerate peft datasets transformers

import os
import torch
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Set up paths matching your Drive organization
DRIVE_BASE = "/content/drive/MyDrive/TrailWatch_AI_Hackthon"

MODEL_PATH = os.path.join(DRIVE_BASE, "Model/Saul-7B-Instruct-v1")
LORA_PATH  = os.path.join(DRIVE_BASE, "Model/Saul-7B-Instruct-v1-lora-adapter") # Path to saved LoRA weights
DATA_PATH  = os.path.join(DRIVE_BASE, "Data/test_dataset.csv")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Configure 4-bit NF4 quantization to optimize RAM/VRAM
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading tokenizer from: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

print(f"Loading base model from: {MODEL_PATH}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=quant_config,
    device_map="auto"
)

print("Base model loaded successfully!")

Loading tokenizer from: /content/drive/MyDrive/TrailWatch_AI_Hackthon/Model/Saul-7B-Instruct-v1
Loading base model from: /content/drive/MyDrive/TrailWatch_AI_Hackthon/Model/Saul-7B-Instruct-v1


/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Base model loaded successfully!


In [ ]:
from datasets import load_dataset

# Load evaluation dataset
# If loading from local CSV/JSON saved on Drive:
eval_df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(eval_df)} evaluation samples.")

def format_prompt(quote):
    # Match the prototype's training/inference format exactly.
    prompt = (
        "### Instruction:\n"
        "Analyze the following trial quote for human rights "
        "violations with the legal basis of ICCPR.\n\n"
        "### Input:\n"
        f"{quote}\n\n"
        "### Response:\n"
    )

    return tokenizer(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)


def generate_analysis(model, inputs, max_new_tokens=1024):
    model.eval()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_length = inputs["input_ids"].shape[-1]
    new_tokens = outputs[0][input_length:]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    print(f"Generated tokens: {new_tokens.numel()}")
    if new_tokens.numel() >= max_new_tokens:
        print("Warning: output reached the token limit.")

    if "Legal Basis & Analysis:" not in response:
        print("Warning: expected analysis heading is missing.")

    return response

Loaded 12 evaluation samples.


In [ ]:
# Select a sample quote from your evaluation dataset
sample_quote = eval_df.iloc[0]['trail_quote']
sample_inputs = format_prompt(sample_quote)

print("--- [BASELINE MODEL ANALYSIS] ---")
baseline_response = generate_analysis(base_model, sample_inputs)
print(baseline_response)

--- [BASELINE MODEL ANALYSIS] ---
Generated tokens: 372
The response should be in the following format:

{
  "violation": "violation type",
  "legal_basis": "legal basis",
  "description": "description of the violation"
}

### Explanation:
The input is a trial quote that contains two separate events. The first event is on August 11, and the second event is on August 13. Both events are related to protests against the government and are calling for the ousting of the current leader, Prayut. The protests are peaceful and non-violent in nature.

The legal basis for this trial quote is the International Covenant on Civil and Political Rights (ICCPR). The ICCCPR is an international human rights treaty that aims to protect the civil and political rights of individuals. The violations in this case are related to the right to peaceful assembly and the right to freedom of expression.

The first event on August 11 violates the right to peaceful assembly because the protestors are calling for a m

In [ ]:
from peft import PeftModel

print(f"Loading fine-tuned LoRA adapter from: {LORA_PATH}")
# Wrap the base model with the trained LoRA adapter weights
ft_model = PeftModel.from_pretrained(base_model, LORA_PATH)
ft_model.eval()

print("Fine-tuned TrailWatch model ready!")

Loading fine-tuned LoRA adapter from: /content/drive/MyDrive/TrailWatch_AI_Hackthon/Model/Saul-7B-Instruct-v1-lora-adapter
Fine-tuned TrailWatch model ready!


In [ ]:
print("--- [FINE-TUNED TRAILWATCH MODEL ANALYSIS] ---")
ft_response = generate_analysis(ft_model, sample_inputs)
print(ft_response)

--- [FINE-TUNED TRAILWATCH MODEL ANALYSIS] ---
Generated tokens: 92
Human Rights Violations: ['Freedom of Expression (Article 19 ICCPR)']
Legal Basis & Analysis: ['Political slogans and calls to peaceful protest are protected under Article 19 of the ICCPR. Criminalizing political speech and peaceful assembly under the Computer Crime Act (CCA) for political expression or organizing protests fails the strict necessity and proportionality tests.']


In [ ]:
row = eval_df.iloc[0]
sample_quote = row["trail_quote"]
sample_inputs = format_prompt(sample_quote)

print("--- ACTUAL MODEL INPUT ---")
print(tokenizer.decode(
    sample_inputs["input_ids"][0],
    skip_special_tokens=False
))

print("\n--- FINE-TUNED RESPONSE ---")
ft_response = generate_analysis(ft_model, sample_inputs)
print(ft_response)

print("\n--- REFERENCE RESPONSE ---")
print(f"Human Rights Violations: {row['human_right_violation']}")
print(f"Legal Basis & Analysis: {row['legal_basis&analysis']}")

--- ACTUAL MODEL INPUT ---
<s> ### Instruction:
Analyze the following trial quote for human rights violations with the legal basis of ICCPR.

### Input:
["11 August Mob to Oust the Tyrant. Starting at 15:00. Meet at the Victory Monument, Phaya Thai side. We will march to Prayut's house. #Thalufah #11AugustMob #Oust Prayut #PeopleRevolution", "Assemble! Friday the 13th. Oust the Tyrant. #13AugustMob. Victory Monument, Phaya Thai side. Starting at 15:00. We will link our arms together and march to Prayut's house... Brothers and sisters, please come out so we have more people to fight together in peace and in a non-violent way to confront those tyrants and bring back democracy!"]

### Response:


--- FINE-TUNED RESPONSE ---
Generated tokens: 114
Human Rights Violations: ['Freedom of Expression (Article 19 ICCPR)', 'Freedom of Assembly (Article 21 ICCPR)']
Legal Basis & Analysis: ['Political speech and peaceful assembly are protected under Articles 19 and 21 of the ICCPR. Criminalizing pol